# 🎮 Introdução à Teoria dos Jogos — Minimax

Este notebook explora os fundamentos da teoria dos jogos aplicada ao projeto
**"21 Jogos Lógicos no Mesmo Tabuleiro"**.

Vamos entender como a IA decide suas jogadas usando o algoritmo **Minimax com poda Alpha-Beta**.

## 1. O que é Minimax?

Minimax é um algoritmo de busca para jogos de **soma zero** com **dois jogadores**.

- **Jogador 1 (Maximizador):** Quer maximizar a pontuação
- **Jogador 2 (Minimizador):** Quer minimizar a pontuação

O algoritmo simula todas as jogadas possíveis em uma árvore de decisão,
alternando entre maximizar e minimizar a cada nível.

In [ ]:
# Importar os módulos do backend
import sys
sys.path.insert(0, '../backend')

from games.tic_tac_toe import TicTacToe
from engine.minimax_ai import MinimaxAI, DIFFICULTY

game = TicTacToe()
print(f'Jogo: {game.name}')
print(f'Descrição: {game.description}')
print(f'Origem: {game.origin}')

## 2. Estado Inicial e Movimentos Válidos

In [ ]:
state = game.get_initial_state()
print('Estado inicial:')
print(f'  Tabuleiro: {state["board"]}')
print(f'  Jogador atual: {state["current_player"]}')
print(f'  Movimentos válidos: {game.get_valid_moves(state)}')
print(f'  Resultado: {game.check_result(state)}')

## 3. Simulando uma Partida

Vamos jogar uma partida completa entre duas IAs com diferentes profundidades.

In [ ]:
def print_board(board):
    """Exibe o tabuleiro de forma visual."""
    symbols = {0: '·', 1: 'X', 2: 'O'}
    for row in range(3):
        cells = [symbols[board[row * 3 + col]] for col in range(3)]
        print(' ' + ' │ '.join(cells))
        if row < 2:
            print(' ──┼───┼──')
    print()

# Partida: IA fácil (depth=1) vs IA difícil (depth=9)
game = TicTacToe()
state = game.get_initial_state()

ai_easy = MinimaxAI(game, max_depth=1, randomize=True)
ai_hard = MinimaxAI(game, max_depth=9, randomize=False)

print('=== Partida: IA Fácil (X) vs IA Difícil (O) ===')
print_board(state['board'])

move_num = 0
while not game.check_result(state)['over']:
    move_num += 1
    current = state['current_player']
    ai = ai_easy if current == 1 else ai_hard
    label = 'Fácil (X)' if current == 1 else 'Difícil (O)'
    
    move = ai.get_best_move(state)
    state = game.apply_move(game.clone_state(state), move)
    
    print(f'Jogada {move_num} — {label}: posição {move}')
    print_board(state['board'])

result = game.check_result(state)
if result['winner'] == 1:
    print('🏆 IA Fácil venceu!')
elif result['winner'] == 2:
    print('🏆 IA Difícil venceu!')
else:
    print('🤝 Empate!')

## 4. Profundidade vs Qualidade da Decisão

Vamos medir como a profundidade do minimax afeta a qualidade das decisões.

In [ ]:
import time

game = TicTacToe()

# P1 (X) quase ganhando — pode vencer jogando na posição 2
test_state = {
    'board': [1, 1, 0, 2, 2, 0, 0, 0, 0],
    'current_player': 1,
}

print('Tabuleiro de teste (P1 pode vencer):')
print_board(test_state['board'])

print(f'{"Depth":>8} {"Move":>6} {"Tempo (ms)":>12} {"Correto?":>10}')
print('-' * 40)

for depth in [1, 2, 3, 5, 9]:
    ai = MinimaxAI(game, max_depth=depth, randomize=False)
    
    t0 = time.perf_counter()
    move = ai.get_best_move(test_state)
    elapsed = (time.perf_counter() - t0) * 1000
    
    correct = '✅' if move == 2 else '❌'
    print(f'{depth:>8} {move:>6} {elapsed:>10.2f}ms {correct:>10}')

## 5. Poda Alpha-Beta

A poda alpha-beta elimina ramos da árvore que **não podem afetar o resultado final**.

- **Alpha:** Melhor valor que o maximizador pode garantir
- **Beta:** Melhor valor que o minimizador pode garantir
- Quando `beta ≤ alpha`, o ramo é **podado** (não precisa ser explorado)

Isso reduz dramaticamente o número de estados avaliados, especialmente em árvores profundas.

In [ ]:
# Demonstração: contar nós visitados com e sem poda
import math

class CountingMinimaxAI(MinimaxAI):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.nodes_visited = 0
    
    def _minimax(self, state, depth, alpha, beta, maximizing):
        self.nodes_visited += 1
        return super()._minimax(state, depth, alpha, beta, maximizing)

game = TicTacToe()
state = game.get_initial_state()

# Com poda alpha-beta
ai_pruned = CountingMinimaxAI(game, max_depth=9, randomize=False)
move = ai_pruned.get_best_move(state)
print(f'Com poda Alpha-Beta: {ai_pruned.nodes_visited:,} nós visitados')

# Sem poda (alpha=-inf, beta=+inf nunca cortam — simular trocando a condição)
# Nota: a árvore completa do Jogo da Velha tem ~549.946 nós
print(f'\nÁrvore completa (sem poda): ~549.946 nós')
print(f'Redução: ~{(1 - ai_pruned.nodes_visited / 549946) * 100:.1f}%')

## 6. Resumo

| Conceito | Descrição |
|---|---|
| **Minimax** | Busca exaustiva alternando max/min |
| **Alpha-Beta** | Poda que elimina ramos irrelevantes |
| **Profundidade** | Controla qualidade vs velocidade |
| **Evaluate** | Heurística: +1000 (P1 vence), -1000 (P2 vence), 0 (empate/em andamento) |
| **Randomize** | Embaralhar movimentos para variedade em profundidades baixas |

No próximo notebook, vamos **visualizar a árvore de decisão** para entender
exatamente como a IA pensa.